# Lebanon Outing Planner: run everything in Colab
**EECE 503P/798S · C5 – Agent Tools · Adam Hijazi**

MCP server (FastMCP, 3 tools) + OpenAI Agents SDK agent (Gemini).

**Before running:** add your Gemini key as a Colab secret named `GEMINI_API_KEY` (🔑 icon in the left sidebar → turn on *Notebook access*). Then `Runtime → Run all`. No GPU is needed.

In [ ]:
# 1. Get the code and install the requirements
import os
if not os.path.exists("c5_mcp_toolkit"):
    !git clone -q https://github.com/adammhijazi/c5_mcp_toolkit.git
%cd c5_mcp_toolkit
!pip -q install -r requirements.txt

In [ ]:
# 2. API key from Colab secrets (never printed)
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", bool(os.environ["GEMINI_API_KEY"]))

## 3. Tests: every tool, every failure case, the MCP protocol, and the full agent round trip (no API key needed)

In [ ]:
!python -m pytest -v

## 4. Live check of the Data tool against the real Open-Meteo API

In [ ]:
!python -c "import server; print(server.fetch_forecast('Beirut', 2).model_dump_json(indent=1))"

## 5. Scripted demo: Data → Action → Orchestration → failure case → resource
Each step shows the round trip: 🔧 agent → MCP client → server, then 📦 server → MCP client → agent, then 🤖 the agent's answer.
(If step 4 failed because the weather API was unreachable, change the command to `!python -u agent.py --demo --offline 2>&1 | cat`.)

The demo starts with an empty outings list each time, so you can re-run it. If a Gemini model is out of free quota, the agent prints why and switches to the next one.

(`2>&1 | cat` makes Colab show every line; without it the server's coloured log can hide the rest of the output.)

In [ ]:
!python -u agent.py --demo 2>&1 | cat

## 6. Chat with the agent
The MCP server stays connected; call `await chat("...")` as many times as you like.

In [ ]:
import sys
import agent as A
from agents import Agent
from agents.mcp import MCPServerStdio
import os, pathlib

# fresh outings file for this chat, so re-running never hits "already saved"
chat_file = pathlib.Path("data/chat_outings.json"); chat_file.unlink(missing_ok=True)
os.environ["OUTINGS_FILE"] = str(chat_file.resolve())

server = MCPServerStdio(name="lebanon-outing-planner",
                        params={"command": sys.executable, "args": [A.SERVER_PATH], "env": dict(os.environ)},
                        cache_tools_list=True, client_session_timeout_seconds=30)
await server.connect()
print("Tools:", [t.name for t in await server.list_tools()])
history = []

async def chat(question):
    global history
    result, used = await A.ask(lambda m: Agent(name="Outing Planner", instructions=A.INSTRUCTIONS,
                                               mcp_servers=[server], model=m), question, history)
    A.print_round_trip(result)
    print(f"🤖 ({used}) {result.final_output}")
    history = result.to_input_list()

await chat("Is it a good week for the beach in Tyre?")

In [ ]:
await chat("Then plan me a hike in Faraya instead.")

In [ ]:
# See what's been saved (MCP resource)
print((await server.read_resource("outings://saved")).contents[0].text)